In [1]:
if "moved_up_dir" not in globals():
    # Code that should run once
    %cd ..
    moved_up_dir = True
else:
    print("Skipping — already executed this session.")

/home/woodbkb2/git/nepo-music


In [2]:
import json
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from apis import spotify as sp
from base_model import features_calculator as fc
from tqdm import tqdm

load_dotenv()

True

In [4]:

DATA_DIR = Path('./data')
ARTIST_FULL_DATA = DATA_DIR / "artist_full_data"
OUTPUT_DIR = DATA_DIR / "artist_filtered_data" 

EXCLUDE_LIST = {'[unknown]'}
    

In [8]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
failures = 0
processed = 0

files = [f for f in ARTIST_FULL_DATA.iterdir() if f.suffix == ".jsonl"]

outer_bar = tqdm(files, desc="Files", unit="file", position=0)

for i, file in enumerate(outer_bar):
    try:
        with file.open() as f, open(OUTPUT_DIR / file.name, 'w') as of:
            # new inner bar, always in the same row (position=1)
            inner_bar = tqdm(
                f,
                desc=f"{file.name}",
                unit="row",
                leave=False,
                position=1,
            )

            for raw_artist_work_data in inner_bar:
                try:
                    processed += 1
                    artist_work_data = json.loads(raw_artist_work_data)

                    if artist_work_data["artist_name"] in EXCLUDE_LIST:
                        continue

                    if len(artist_work_data['works']) == 0:
                        continue
                    
                    of.write(raw_artist_work_data + '\n')

                    if processed % 1000 == 0:
                        # update the outer bar info instead of printing a new line
                        outer_bar.set_postfix(
                            processed=processed,
                            failures=failures,
                        )

                except Exception as e:
                    failures += 1
                    outer_bar.set_postfix(
                        processed=processed,
                        failures=failures,
                        last_error="row",
                    )

            inner_bar.close()

    except Exception as e:
        failures += 1
        outer_bar.set_postfix(
            processed=processed,
            failures=failures,
            last_error=f"file {file.name}",
        )
        print(e)

outer_bar.close()


Files:  11%|█         | 21/194 [03:42<31:26, 10.90s/file, failures=0, processed=35000]